In [6]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

In [5]:
#a)

X_fantoma = np.array([
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
[0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
[0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
[0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)

x_fantoma = X_fantoma.reshape(-1)

#vuelvo a crear las matrices A2,A4,A7
def mascara_rayo(n, direccion, s):
    res =  np.zeros((n, n))
    for i in range(n) :
        for j in range(n):
            if (direccion[0]*i + direccion[1]*j) == s:
                res[i,j] =1
    return res
            

def matriz_proyeccion(n, direcciones):
    filas_A = []
    for direccion in direcciones:
        valoresdes=[]
        for i in range(n):
            for j in range(n):
                s = direccion[0]*i + direccion[1]*j
                valoresdes.append(s)
    
        ssinrepetir = []

        for k in valoresdes:
            if k not in ssinrepetir:
                ssinrepetir.append(k)        

        ssinrepetir.sort()

        for s in ssinrepetir:
            fila_s =mascara_rayo(n, direccion, s)
            filas_A.append(fila_s.reshape(-1))
           
    A=np.array(filas_A)
    return A

d2 = [(1,0), (0,1)]
d4 = [(1,0), (0,1), (1,1), (1, -1)]
d7 = [(1,0), (0,1), (1,1), (1, -1), (2,1),(1,2),(3,1)]


A2 = matriz_proyeccion(8, d2)
A4 = matriz_proyeccion(8, d4)
A7= matriz_proyeccion(8, d7)

def generar_mediciones(A, x):
    return A @ x

b2 = generar_mediciones(A2, x_fantoma)
b4 = generar_mediciones(A4, x_fantoma)
b7 = generar_mediciones(A7, x_fantoma)

#a partir de las Ak y bk busco el xk que deberia ser x_fantoma, resuelvo el sistema 
#Si el sistema tiene infinitas soluciones, Gauss-Jordan te muestra la familia de soluciones con variables libres.
#lstsq, en cambio, elige una sola: la solución de norma mínima.
#Y si el sistema es incompatible por ruido, Gauss-Jordan no encuentra una solución exacta,
# mientras que lstsq devuelve la que mejor aproxima a b , minimizando: (la norma) ∥Ax−b∥
def reconstruir(A, b):
    x_hat, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
    return x_hat

x2_hat = reconstruir(A2, b2)
x4_hat = reconstruir(A4, b4)
x7_hat = reconstruir(A7, b7)



In [11]:
#b)

def residuo(Ak, bk, xk_hat, x_fantoma):
    resk = bk - Ak@xk_hat
    norma_resk = np.linalg.norm(resk)
    error_relativo = np.linalg.norm(xk_hat - x_fantoma) / np.linalg.norm(x_fantoma)
    return norma_resk, error_relativo

res2= residuo(A2, b2, x2_hat, x_fantoma)
res4= residuo(A4, b4, x4_hat, x_fantoma)
res7= residuo(A7, b7, x7_hat, x_fantoma)

tabla = pd.DataFrame({
    "matriz": ["A2", "A4", "A7"],
    "rango": [np.linalg.matrix_rank(A2), np.linalg.matrix_rank(A4), np.linalg.matrix_rank(A7)],
    "norma_residuo": [res2[0], res4[0], res7[0]],
    "error_relativo": [res2[1], res4[1], res7[1]]
})

print(tabla)


  matriz  rango  norma_residuo  error_relativo
0     A2     15   6.658851e-15    4.689265e-01
1     A4     39   7.996052e-15    1.364605e-01
2     A7     64   1.986808e-14    2.550078e-15
